# 📊 01 — Dataset Profiling

**Profiling 6 datasets** สำหรับ Customer Financial Dimension

---

## 📋 Datasets

| # | Dataset | Theme |
|---|---------|-------|
| 01 | Credit Risk Assessment | Risk Score |
| 02 | Loan Exposure | Loan Balance |
| 03 | Delinquency Profile | DQ History |
| 04 | Repayment Consistency | Repayment Pattern |
| 05 | Protection Coverage | PPI Policy |
| 06 | Customer Maturity | Tenure |

---

## 📖 วิธีอ่าน Notebook นี้

| สัญลักษณ์ | ความหมาย |
|-----------|----------|
| 📖 | อ่านเท่านั้น |
| ⚙️ | ต้องรัน |
| ⚠️ | ระวัง |
| ✅ | เสร็จแล้ว |

---

## 1. Setup

👉 **Action Required:** รัน Cell ถัดไป

In [ ]:
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append(str(Path.cwd().parent))
from src import data_loader, profiling

# Setup style
sns.set_theme(style='darkgrid')
plt.rcParams['figure.figsize'] = (12, 6)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

print('✅ Setup complete')

---

## 2. Load All Datasets

👉 **Action Required:** รัน Cell ถัดไปเพื่อโหลด 6 datasets

In [ ]:
# Load config
config = data_loader.load_config('../config.yaml')

# Load all datasets
datasets = {}
data_files = {
    'credit_risk':     '01_Credit_Risk_Assessment.csv',
    'loan_exposure':   '02_Loan_Exposure.csv',
    'delinquency':     '03_Delinquency_Profile.csv',
    'repayment':       '04_Repayment_Consistency.csv',
    'protection':      '05_Protection_Coverage.csv',
    'maturity':        '06_Customer_Maturity.csv',
}

data_dir = Path('../data/raw')

for name, filename in data_files.items():
    filepath = data_dir / filename
    if filepath.exists():
        try:
            df = pd.read_csv(filepath, encoding='utf-8-sig')
            datasets[name] = df
            print(f'✅ {name:20s} {df.shape[0]:>5} rows × {df.shape[1]:>3} cols')
        except Exception as e:
            print(f'⚠️ {name:20s} Error: {e}')
    else:
        print(f'❌ {name:20s} Not found: {filepath}')

print()
print(f'📁 Loaded {len(datasets)} / 6 datasets')

---

## 3. Dataset Summary

📖 **Read Only:** ดูภาพรวม

In [ ]:
# Summary table
summary = data_loader.get_dataset_summary(datasets)
summary

---

## 4. Profile Each Dataset

👉 **Action Required:** รัน Cell ถัดไปเพื่อดู profile ของแต่ละ dataset

In [ ]:
# Profile function
def profile_df(df, name):
    print('=' * 80)
    print(f'📊 DATASET: {name.upper()}')
    print('=' * 80)
    print(f'Shape: {df.shape[0]} rows × {df.shape[1]} cols')
    print(f'Memory: {df.memory_usage(deep=True).sum() / 1024:.2f} KB')
    print()
    
    # Column info
    col_info = pd.DataFrame({
        'dtype': df.dtypes.astype(str),
        'non_null': df.notna().sum(),
        'null': df.isna().sum(),
        'null_pct': (df.isna().sum() / len(df) * 100).round(2),
        'unique': df.nunique(),
    })
    print('Columns:')
    print(col_info.to_string())
    print()

# Profile each dataset
for name, df in datasets.items():
    profile_df(df, name)

---

## 5. Data Quality Check

👉 **Action Required:** รัน Cell ถัดไปเพื่อตรวจสอบคุณภาพข้อมูล

In [ ]:
# Data quality check
for name, df in datasets.items():
    print('=' * 80)
    print(f'🔍 DATA QUALITY: {name.upper()}')
    print('=' * 80)
    quality = profiling.check_data_quality(df)
    issues = quality[quality['issues'] != 'OK']
    if len(issues) > 0:
        print(issues.to_string(index=False))
    else:
        print('✅ No major issues')
    print()

---

## 6. CNNULL Analysis

📖 **Read Only:** CNNULL = จำนวน column ที่ไม่ null (Data Completeness Indicator)

👉 **Action Required:** รัน Cell ถัดไป

In [ ]:
# CNNULL analysis
for name, df in datasets.items():
    if 'CNNULL' in df.columns:
        print('=' * 80)
        print(f'📊 CNNULL: {name.upper()}')
        print('=' * 80)
        summary = profiling.summarize_cnn_null(df)
        print(summary.to_string(index=False))
        print()

---

## 7. Missing Value Heatmap

👉 **Action Required:** รัน Cell ถัดไปเพื่อ plot

In [ ]:
# Missing value heatmap
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, (name, df) in enumerate(datasets.items()):
    if i >= 6:
        break
    
    # Missing matrix
    missing = df.isna()
    sns.heatmap(missing, cbar=False, ax=axes[i], cmap='RdYlGn_r')
    axes[i].set_title(f'{name}\n({df.shape[0]} rows)', fontsize=11, fontweight='bold')
    axes[i].set_xlabel('')
    axes[i].set_ylabel('')

plt.suptitle('Missing Value Heatmap (Red = Missing)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/figures/missing_value_heatmap.png', dpi=100, bbox_inches='tight')
plt.show()

print('✅ Saved: outputs/figures/missing_value_heatmap.png')

---

## 8. Next Steps

📖 **Read Only:** ขั้นถัดไป

| ขั้น | Notebook |
|------|----------|
| สร้าง Data Catalog | `02_Data_Catalog.ipynb` |
| สร้าง Features | `03_Feature_Engineering.ipynb` |
| สร้างภาพ | `05_Visualization.ipynb` |